<h2>Description</h2>

Dans ce code, nous allons établir un modèle afin de prédire le débit horaire sur les Champs Élysées.

Imports

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.inspection import permutation_importance

doc = 'sts_peres.csv'

df_final = pd.read_csv('../datasets_axes_with_all_features/'+ doc, sep=';')

In [2]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9644 entries, 0 to 9643
Columns: 1579 entries, Unnamed: 0 to lag_or_35_23
dtypes: float64(1564), int64(1), object(14)
memory usage: 116.2+ MB


In [3]:
df_final = df_final.copy()
df_final['Date et heure de comptage'] = pd.to_datetime(df_final['Date et heure de comptage'], errors='coerce')
df_final = df_final.sort_values('Date et heure de comptage').reset_index(drop=True)

for col in ['est_vacances', 'est_ferie', 'est_avant_ferie', 'est_pieton']:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

features = [
    'Température', 'precipitations heure',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos', 
    'force moyenne vent (m/s)', 'jour_semaine', 'est_weekend', 'est_vacances', 'est_avant_vacances',
    'est_ferie', 'est_avant_ferie', 'est_pieton','est_rentree'
]

target = 'Débit horaire'

mask_known   = df_final[target].notna()
mask_missing = df_final[target].isna()

X_known = df_final.loc[mask_known, features].copy()
y_known = df_final.loc[mask_known, target].astype(float)
X_missing = df_final.loc[mask_missing, features].copy()

numeric_features = [
    'Température', 'precipitations heure',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos',
    'force moyenne vent (m/s)','est_weekend', 'est_vacances', 'est_avant_vacances',
    'est_ferie', 'est_avant_ferie', 'est_pieton', 'est_rentree'
]

categorical_features = ['jour_semaine']

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)  
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)         

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', ohe)
        ]), categorical_features),
    ],
    remainder='drop'
)

model = HistGradientBoostingRegressor(
    loss='absolute_error',   
    max_depth=18,
    max_iter=70,
    early_stopping=False,
    random_state=42
)

pipe = Pipeline(steps=[('prep', preprocess), ('model', model)])

# ---------- 3) Split chronologique ----------
X_train, X_test, y_train, y_test = train_test_split(
    X_known, y_known, test_size=0.2, shuffle=False
)

# ---------- 4) Pondérations (férié / veille / piéton) ----------
W_FERIE   = 3.0
W_AVANT   = 1.5
W_PIETON  = 10
W_RENTREE = 10
POST_SCALE_FERIE = 1.0

def make_weights(X_frame):
    w = np.ones(len(X_frame), dtype=float)
    is_ferie  = X_frame['est_ferie'].fillna(0).astype(int).to_numpy()
    is_avant  = X_frame['est_avant_ferie'].fillna(0).astype(int).to_numpy()
    is_pieton = X_frame['est_pieton'].fillna(0).astype(int).to_numpy()
    is_rentree = X_frame['est_rentree'].fillna(0).astype(int).to_numpy()

    w[is_ferie == 1]  = W_FERIE
    w[is_avant == 1]  = np.maximum(w[is_avant == 1], W_AVANT)
    w[is_pieton == 1] = W_PIETON
    w[is_rentree == 1] = W_RENTREE

    # Option : normalisation pour garder une échelle de perte comparable
    #w *= (len(w) / w.sum())
    return w

w_train = make_weights(X_train)

# ---------- 5) Entraînement ----------
pipe.fit(X_train, y_train, model__sample_weight=w_train)

# ---------- 6) Prédiction + post-ajustement éventuel ----------
y_pred = pipe.predict(X_test)

mask_ferie_test = X_test['est_ferie'].fillna(0).astype(int).to_numpy() == 1
mask_est_pieton_test = X_test['est_pieton'].fillna(0).astype(int).to_numpy() == 1
y_pred[mask_ferie_test] *= POST_SCALE_FERIE
#y_pred[mask_est_pieton_test] *= 0.5

# ---------- 7) Évaluation ----------
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² : {r2:.3f}")
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")

# Diagnostics par sous-régimes
is_pieton_test = X_test['est_pieton'].fillna(0).astype(int) == 1
print(f"Part d'observations piéton (test) : {is_pieton_test.mean():.1%}")
if is_pieton_test.any():
    mae_pieton = mean_absolute_error(y_test[is_pieton_test], y_pred[is_pieton_test])
    print(f"MAE (jours piéton) : {mae_pieton:.2f} (n={is_pieton_test.sum()})")
    mae_non_pieton = mean_absolute_error(y_test[~is_pieton_test], y_pred[~is_pieton_test])
    print(f"MAE (jours non piéton) : {mae_non_pieton:.2f} (n={(~is_pieton_test).sum()})")

df_final['Débit_prédit'] = np.nan
df_final.loc[X_test.index, 'Débit_prédit'] = y_pred


R² : 0.892
MAE : 53.41
RMSE : 67.63
Part d'observations piéton (test) : 0.0%


In [4]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9644 entries, 0 to 9643
Columns: 1580 entries, Unnamed: 0 to Débit_prédit
dtypes: datetime64[ns, UTC](1), float64(1565), int64(1), object(13)
memory usage: 116.3+ MB


In [5]:
from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np
import plotly.express as px

# 1) Importance par permutation sur le pipeline complet (prétraitements inclus)
perm = permutation_importance(
    estimator=pipe,
    X=X_test,
    y=y_test,
    n_repeats=20,
    random_state=42,
    scoring='neg_root_mean_squared_error'  # cohérent avec votre RMSE
)

imp_df = (
    pd.DataFrame({
        'feature': features,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std
    })
    .sort_values('importance_mean', ascending=False)
)

print(imp_df.head(20))

# 2) Bar chart Plotly (top 20)
topk = imp_df.head(20).sort_values('importance_mean', ascending=True)
fig = px.bar(
    topk,
    x='importance_mean', y='feature',
    error_x='importance_std',
    orientation='h',
    title='Importance par permutation — Top 20 (plus haut = plus influent)'
)
fig.update_layout(xaxis_title="Perte de performance (Δ RMSE, signe inversé)", yaxis_title="")
fig.show()


                     feature  importance_mean  importance_std
2                  heure_sin       186.390259       10.166707
3                  heure_cos        89.026076        5.267127
4                   jour_sin        30.673016        3.408866
5                   jour_cos         8.562752        2.299397
0                Température         3.323902        1.395559
11              est_vacances         1.903315        0.577691
6                   mois_sin         1.544226        0.938368
10               est_weekend         0.662677        0.282626
9               jour_semaine         0.597712        0.336014
8   force moyenne vent (m/s)         0.551969        0.587405
13                 est_ferie         0.505740        0.364385
14           est_avant_ferie         0.005756        0.008846
7                   mois_cos         0.000000        0.000000
12        est_avant_vacances         0.000000        0.000000
15                est_pieton         0.000000        0.000000
16      

In [6]:
time_index = df_final.loc[X_test.index, 'Date et heure de comptage']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_pred,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_test,
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()

In [7]:
X_all = df_final.loc[:, features].copy()

predictions_all = pipe.predict(X_all)

serie = pd.Series(predictions_all, index=df_final.index)

commun = df_final[target].notna() & serie.notna()

predictions_with_know = serie.loc[commun].astype(float)

mae = mean_absolute_error(predictions_with_know, y_known)
rmse = np.sqrt(mean_squared_error(predictions_with_know, y_known))
r2   = r2_score(predictions_with_know, y_known)

print("Nombre de valeurs : " + str(len(predictions_with_know)))
print(f"R² : {r2:.3f}")
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")

df_final['débit_prédit_all'] = predictions_all 


Nombre de valeurs : 1404
R² : 0.929
MAE : 35.94
RMSE : 57.50


In [8]:
df_final['débit_prédit_all'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 9644 entries, 0 to 9643
Series name: débit_prédit_all
Non-Null Count  Dtype  
--------------  -----  
9644 non-null   float64
dtypes: float64(1)
memory usage: 75.5 KB


In [9]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_final['Date et heure de comptage'],
    y=predictions_all,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=df_final['Date et heure de comptage'],
    y=df_final['Débit horaire'],
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()

<h2>Taux d'occupation</h2>

In [10]:
target_occ = 'Taux d\'occupation'

features_occ = [
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin',  
    'est_weekend', 'est_vacances', 'est_avant_vacances',
    'est_ferie', 'est_avant_ferie', 'est_pieton',
    'lag_or_4_1', 'lag_or_5_0', 'lag_or_5_1', 'lag_or_4_23',
    'lag_or_6_23', 'lag_or_7_0', 'lag_or_7_1', 'est_rentree'
]

mask_occ = df_final[target_occ].notna()
mask_missing_occ = df_final[target_occ].isna()

X_occ = df_final.loc[mask_occ, features_occ].copy()
y_occ = df_final.loc[mask_occ, target_occ].astype(float)

X_train_occ, X_test_occ, y_train_occ, y_test_occ = train_test_split(
    X_occ, y_occ, test_size=0.18, shuffle=False
)

numeric_features_occ = [c for c in features_occ if c != 'jour_semaine']
categorical_features_occ = []

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocess_occ = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features_occ),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', ohe)
        ]), categorical_features_occ),
    ],
    remainder='drop'
)

model_occ = HistGradientBoostingRegressor(
    loss='poisson',
    max_depth=8,
    max_iter=1000,
    early_stopping=False,
    random_state=42
)

pipe_occ = Pipeline(steps=[('prep', preprocess_occ), ('model', model_occ)])

pipe_occ.fit(X_train_occ, y_train_occ)

y_pred_occ = pipe_occ.predict(X_test_occ)

print("=== Performances taux d'occupation ===")
print(f"R²   : {r2_score(y_test_occ, y_pred_occ):.3f}")
print(f"MAE  : {mean_absolute_error(y_test_occ, y_pred_occ):.2f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test_occ, y_pred_occ)):.2f}")
print(f"nb test : {len(y_test_occ)}")


=== Performances taux d'occupation ===
R²   : 0.438
MAE  : 1.72
RMSE : 2.41
nb test : 253


In [11]:
perm = permutation_importance(
    estimator=pipe_occ,
    X=X_test_occ,
    y=y_test_occ,
    n_repeats=20,
    random_state=42,
    scoring='neg_root_mean_squared_error'  
)

imp_df = (
    pd.DataFrame({
        'feature': features_occ,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std
    })
    .sort_values('importance_mean', ascending=False)
)

print(imp_df.head(20))

topk = imp_df.head(20).sort_values('importance_mean', ascending=True)
fig = px.bar(
    topk,
    x='importance_mean', y='feature',
    error_x='importance_std',
    orientation='h',
    title='Importance par permutation — Top 20 (plus haut = plus influent)'
)
fig.update_layout(xaxis_title="Perte de performance (Δ RMSE, signe inversé)", yaxis_title="")
fig.show()


               feature  importance_mean  importance_std
0            heure_sin         1.620572        0.159951
1            heure_cos         0.728516        0.091536
4             mois_sin         0.370340        0.112772
16          lag_or_7_0         0.154198        0.032788
6         est_vacances         0.143052        0.072814
17          lag_or_7_1         0.133267        0.039976
3             jour_cos         0.085419        0.037212
2             jour_sin         0.012244        0.074550
13          lag_or_5_1         0.012046        0.026229
10          est_pieton         0.000000        0.000000
18         est_rentree         0.000000        0.000000
7   est_avant_vacances         0.000000        0.000000
9      est_avant_ferie        -0.005982        0.004071
15         lag_or_6_23        -0.034931        0.029097
14         lag_or_4_23        -0.035280        0.035266
8            est_ferie        -0.042108        0.035258
5          est_weekend        -0.080337        0

In [12]:
time_index_occ = df_final.loc[X_test_occ.index, 'Date et heure de comptage']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_index_occ,
    y=y_pred_occ,
    mode='lines',
    name="Taux d'occupation prédit"
))
fig.add_trace(go.Scatter(
    x=time_index_occ,
    y=y_test_occ,
    mode='lines',
    name="Taux d'occupation réel"
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Taux d'occupation (%)",
    hovermode='x unified'
)

fig.show()

In [13]:
X_all_occ = df_final.loc[:, features_occ].copy()

predictions_all_occ = pipe_occ.predict(X_all_occ)

serie_occ = pd.Series(predictions_all_occ, index=df_final.index)

commun_occ = df_final[target_occ].notna() & serie_occ.notna()

predictions_with_know_occ = serie_occ.loc[commun_occ].astype(float)

mae = mean_absolute_error(predictions_with_know_occ, y_occ)
rmse = np.sqrt(mean_squared_error(predictions_with_know_occ, y_occ))
r2   = r2_score(predictions_with_know_occ, y_occ)

print("Nombre de valeurs : " + str(len(predictions_with_know_occ)))
print(f"R² : {r2:.3f}")
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")

Nombre de valeurs : 1404
R² : 0.931
MAE : 0.48
RMSE : 1.11


In [14]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_final['Date et heure de comptage'],
    y=predictions_all_occ,
    mode='lines',
    name='Taux prédit'
))
fig.add_trace(go.Scatter(
    x=df_final['Date et heure de comptage'],
    y=df_final[target_occ],
    mode='lines',
    name="Taux d occupation réel"
))

fig.update_layout(
    title="Comparaison des taux occupation (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Taux d'occupation (%)",
    hovermode='x unified'
)

fig.show()